<a id="innhold"></a>

# GNSS Multipath Analysis – bruk av biblioteket direkte

Denne notebooken viser hvordan `gnssmultipath` kan brukes som et vanlig Python-bibliotek, uten å kjøre
hele multipath-analysen: lese RINEX-observasjoner, interpolere satellittkoordinater fra både
kringkastede efemerider og presise baner (SP3), og til slutt visualisere resultatet.

## Innhold

1. [RINEX observasjonsfiler](#rinex-obs)
    - [1.1 Hente ut data](#hente-ut-data)
    - [1.2 Lage en pandas DataFrame](#dataframe)
2. [Interpolasjon av satellittkoordinater](#satkoord)
    - [2.1 Kringkastede efemerider (navigasjonsfiler)](#broadcast)
    - [2.2 Presise satellittkoordinater (SP3-filer)](#sp3)
3. [Diverse – polarplot](#diverse)

Cellene er ment å kjøres i rekkefølge, siden senere seksjoner gjenbruker objekter fra de tidligere.


<a id="rinex-obs"></a>

# 1. RINEX observasjonsfiler

`readRinexObs` leser observasjonsfila og returnerer et `RinexObsData`-objekt med observasjoner,
epoketider og headerinformasjon. Riktig leserutine (RINEX v2 eller v3/v4) velges automatisk ut fra
versjonsnummeret i fila.

[Tilbake til innholdsfortegnelsen](#innhold)


In [ ]:
from pathlib import Path
import pandas as pd
from gnssmultipath import readRinexObs

pd.set_option('display.float_format', '{:,.3f}'.format)

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'TestData').is_dir())
rinObsFilename1 = repo_root / 'TestData' / 'ObservationFiles' / 'v3' / 'OPEC00NOR_S_20220010000_01D_30S_MO_3.04.rnx'

# Hele RinexObsData-objektet beholdes, slik at tid, header og posisjon kan gjenbrukes i seksjon 2
rinex = readRinexObs(rinObsFilename1)

# .observations gir det kodebaserte grensesnittet mot selve observasjonene
obs = rinex.observations


<a id="hente-ut-data"></a>

## 1.1 Hente ut data

Eksempler på hvordan man kan hente ut data for hvert system, samt hvordan man henter enkeltsignaler,
frekvenser, bølgelengder osv.


In [ ]:
# Hent observasjoner for hver GNSS-konstellasjon
gps = obs["G"]
galileo = obs["E"]
glonass = obs["R"]
beidou = obs["C"]

# Hent ut frekvensene for de to første GPS-båndene (L1 og L2) for å kunne lage ionosfærefrie lineærkombinasjoner
f1 = gps.frequency('C1C')
f2 = gps.frequency('C2W')

# Hente ut bølgelengdene for de to første GPS-båndene (L1 og L2)
gps_lambda1 = gps.wavelength('L1C')
gps_lambda2 = gps.wavelength('L2W')


# Pseudoranges i meter, faseobservasjonene konvertert fra bølgelengder til meter
C1C = gps.get('C1C')
C2W = gps.get('C2W')
L1C = gps.get('L1C') * gps_lambda1
L2W = gps.get('L2W') * gps_lambda2


# Ionosfære-fri kodemålinger
P_IF = (f1**2 * C1C - f2**2 * C2W) / (f1**2 - f2**2)

# Ionosfære-fri faseobservasjoner
L_IF = (f1**2 * L1C - f2**2 * L2W) / (f1**2 - f2**2)



<a id="dataframe"></a>

## 1.2 Lage en pandas DataFrame med de signalene man ønsker

Lett å se på de aktuelle signalene, samt skrive til en CSV-fil om ønskelig.


### Kun GPS


In [ ]:
CODES = ['C1C', 'C2W', 'L1C', 'L2W']

# Én rad per satellitt og epoke. Raden beholdes så lenge minst én av kodene
# har en verdi; manglende enkeltsignaler blir NaN i sin egen kolonne.
# Bruk gps.to_dataframe(codes=CODES, dropna=False) for det fulle rutenettet.
df_gps = (gps.to_dataframe(codes=CODES)
           # 'sv' er nullpolstret (G01..G32) og sorterer derfor likt som PRN.
           # Ved flere systemer i samme tabell må 'system' også inn i indeksen.
           .pivot(index=['sv', 'datetime'], columns='code', values='value')[CODES]
           .reset_index()
           .rename_axis(columns=None))  # fjerner 'code' som navn på kolonneaksen

df_gps.to_csv('gps_observations.csv', index=False)
df_gps.head(3)


### Alle systemer og alle observasjonskoder i én tabell

Utelater man `systems` og `codes`, tas alle konstellasjoner og alle koder i fila med.
Kolonnesettet blir da unionen av kodene på tvers av systemene, så en kode som bare
finnes i ett system (f.eks. `C2W` for GPS) blir NaN for de øvrige.


In [ ]:
for sys_code in obs.systems:
    print(f"{obs[sys_code].system_name:9s} ({sys_code}): {obs[sys_code].codes}")

# Uten codes/systems tas alt med. 'system' må være med i indeksen, siden
# PRN-numrene gjentas på tvers av konstellasjonene.
df_alle = (obs.to_dataframe()
             .pivot(index=['system', 'sv', 'datetime'], columns='code', values='value')
             .reset_index()
             .rename_axis(columns=None))

df_alle.to_csv('alle_observasjoner.csv', index=False)
df_alle.head(5)



<a id="satkoord"></a>

# 2. Interpolasjon av satellittkoordinater

Satellittkoordinater i ECEF kan hentes fra to kilder, og begge interpoleres til observasjonsepokene:

| Kilde | Klasse | Nøyaktighet | Kommentar |
| --- | --- | --- | --- |
| Kringkastede efemerider (RINEX nav) | `SatelliteEphemerisToECEF` | ~1–2 m | Følger med signalet, tilgjengelig i sanntid |
| Presise baner (SP3) | `PreciseSatCoords` | ~2–5 cm | Lastes ned i etterkant (f.eks. fra CDDIS) |

Begge klassene gir samme datastruktur tilbake:

```python
{'G': {'position': {'1': array([[X, Y, Z], ...]), ...},
       'azimuth':   array([n_epoker, maks_PRN + 1]),
       'elevation': array([n_epoker, maks_PRN + 1])}, ...}
```



<a id="broadcast"></a>

## 2.1 Kringkastede efemerider (navigasjonsfiler)

Her leses navigasjonsfila inn med `RinexNav`, og `SatelliteEphemerisToECEF` konverterer de
kringkastede efemeridene til ECEF-koordinater og interpolerer dem til observasjonsepokene:

* GPS, Galileo og BeiDou: Kepler-elementene propageres til hver epoke (`Kepler2ECEF`).
* GLONASS: tilstandsvektoren integreres med 4. ordens Runge-Kutta (`GLOStateVec2ECEF`).



In [ ]:
from pathlib import Path
import numpy as np
from gnssmultipath import readRinexObs, RinexNav, SatelliteEphemerisToECEF

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'TestData').is_dir())
rinObsFilename1 = repo_root / 'TestData' / 'ObservationFiles' / 'v3' / 'OPEC00NOR_S_20220010000_01D_30S_MO_3.04.rnx'
rinNavFilename = repo_root / 'TestData' / 'NavigationFiles' / 'v3' / 'BRDC00IGS_R_20220010000_01D_MN.rnx'

# Lese inn observasjonsfila
rinex = readRinexObs(rinObsFilename1)

# Lese inn navigasjonsfila. Dette gir et RinexNav-objekt med alle ephemeridene.
navdata = RinexNav.read_nav(str(rinNavFilename))

# Tilnærmet mottakerposisjon (ECEF) fra headeren i observasjonsfila
x_rec, y_rec, z_rec = rinex.approxPosition.flatten().astype(float)
print(f"Approx. posisjon (ECEF) fra RINEX header: X={x_rec:.3f}  Y={y_rec:.3f}  Z={z_rec:.3f}")


# Vis efemeridene som dataframes. .gps/.glonass/.galileo/.beidou gir en DataFrame med navngitte kolonner per system
print("DataFrame med GPS-efemerider:")
display(navdata.gps.head(3))
print("DataFrame med Glonass-efemerider:")
display(navdata.glonass.head(3))
print("DataFrame med Galileo-efemerider:")
display(navdata.galileo.head(3))
print("DataFrame med BeiDou-efemerider:")
display(navdata.beidou.head(3))


### Interpoler efemeridene til observasjonsepokene

`get_sat_ecef_coordinates` propagerer efemeridene til hver observasjonsepoke og gir
satellittkoordinatene i ECEF. Tida oppgis som time-of-week i sekunder, eller som gregoriansk tid
med `time_fmt='GREGORIAN'`.
```


### Satellittkoordinatene som dictionary
Resultatet er en nøstet dictionary med tre nivåer:

```text
sat_coord
└── [systemkode]             # 'G', 'R', 'E', 'C'
    └── ['position']         # Nøkkel for posisjon
        └── [PRN]            # '1', '12' (uten nullpolstring)
            └──> np.ndarray  # shape: (n_epoker, 3)
```

* **systemkode** – `'G'`, `'R'`, `'E'` eller `'C'`
* **PRN** – satellittnummeret som streng *uten* nullpolstring: `'1'`, `'12'` (ikke `'01'`)
* **verdien** – kolonnene er X, Y og Z i meter, én rad per epoke. `None` for satellitter som ikke
  har efemerider i navigasjonsfila.

Koordinatene er rotert for jordrotasjonen i signalets gangtid, altså uttrykt i ECEF-rammen på
mottakstidspunktet. Det er derfor mottakerposisjonen må oppgis til konstruktøren.

Etter at `compute_satellite_azimut_and_elevation_angle` er kjørt (seksjon 3), får hvert system også
nøklene `'azimuth'` og `'elevation'`.

```python
# Alle systemer og satellitter
sat_coord = converter.get_sat_ecef_coordinates(tow)
sat_coord['G']['position']['12']        # ECEF-koordinater for G12, shape (n_epoker, 3)

In [ ]:
# 'navdata' er allerede lest inn over, så den gjenbrukes her i stedet for å lese fila på nytt.
converter = SatelliteEphemerisToECEF(navdata, x_rec, y_rec, z_rec, desired_systems=['G', 'R', 'E', 'C'])

# Henter observasjonsepokene gitt som [GPS-uke, time-of-week] fra rinex objektet
time_epochs, tow = rinex.time_epochs, rinex.time_epochs[:, 1]


# Interpoler til observasjonsepokene (time-of-week i sekunder)
sat_coord = converter.get_sat_ecef_coordinates(tow, output_format="dict") # Interploerte satellittkoordinater i ECEF for alle systemer og alle epoker.
# `converter.to_dataframe()` gir em DataFrame uten å regne på nytt.

# Dictinary med systemer som nøkler, og for hver systemnøkkel er det en dict med 'position' som nøkler. Hver av disse er en dict med satellitt-ID/PRN som nøkler, og for hver satellitt-ID er det en numpy array med koordinater for alle epoker.
sat_coord['G']["position"]["12"] # Interpolerte ECEF-koordinater for GPS-satellitt G12 for alle epoker. Dette er en numpy array med shape (N, 3), der N er antall epoker


### Satellittkoordinatene som DataFrame

Med `output_format='pd.DataFrame'` returneres koordinatene som en DataFrame med multiindeks
(`timestamp`, `system`, `SV`) i stedet for en dictionary, noe som gjør det enkelt å hente ut
enkeltsystemer eller enkeltsatellitter. `converter.to_dataframe()` gir det samme uten å regne på nytt.

Tida kan oppgis som `datetime64`, altså rett fra `rinex.datetimes`. Oppgir man time-of-week i stedet,
hentes GPS-uka fra efemeridene slik at tidsstemplene i indeksen blir riktige uansett.


In [ ]:
# Samme beregning, men med tidsstempler inn og DataFrame ut
df_sat_coord = converter.get_sat_ecef_coordinates(rinex.datetimes, output_format='pd.DataFrame')

# Hente bare Galileo-satellitter
df_galileo = df_sat_coord.xs('E', level='system')

# Hente en bestemt Galileo-satellitt (f.eks. E01)
df_galileo_e01 = df_sat_coord.xs(('E', 'E01'), level=('system', 'SV'))
display(df_galileo_e01.head(3))
display(df_galileo.head(3))

<a id="sp3"></a>

## 2.2 Presise satellittkoordinater (SP3-filer)

SP3-filer inneholder ferdig beregnede satellittposisjoner i ECEF, typisk hvert 5. eller 15. minutt.
`PreciseSatCoords` leser fila (`SP3Reader`) og interpolerer posisjonene til observasjonsepokene med
Nevilles algoritme (`SP3Interpolator`, 7 punkter som standard).

Klassen tar imot enten et ferdig innlest `RinexObsData`-objekt, en sti til en observasjonsfil, eller
bare epokene (`time_epochs`) hvis man ikke har observasjoner. Mottakerposisjonen trengs ikke for å
interpolere banene, bare for azimut og elevasjon, og oppgis derfor til de metodene.


In [ ]:
from gnssmultipath import PreciseSatCoords, SPEED_OF_LIGHT

sp3Filename = repo_root / 'TestData' / 'SP3' / 'Testfile_20220101.eph'

# 'rinex' er allerede lest inn, så observasjonsfila leses ikke på nytt.
# Flere SP3-filer kan også sendes inn som en liste, f.eks. for å dekke et døgnskille.
precise = PreciseSatCoords(sp3Filename, rinex_obs_file=rinex, GNSSsystems=['G', 'R', 'E', 'C'])

# Interpolerte koordinater som DataFrame med kolonnene Epoch, Satellite, X, Y, Z og Clock Bias
df_precise = precise.satcoords
df_precise.head()


### Sammenligning med den kringkastede banen

Hvor mye bedre er egentlig den presise banen? Under sammenlignes alle satellitter som finnes i begge
løsningene, epoke for epoke.

Begge klassene gir koordinater i ECEF-rammen på mottakstidspunktet, altså rotert for jordrotasjonen i
signalets gangtid. SP3-koordinatene gjelder for selve epoken, og roteres derfor tilsvarende før
sammenligningen. Uten denne rotasjonen blir avviket 140–175 m, som er ren rammeforskjell og ikke
banefeil.

Forventet resultat er 1–3 m i middel, som er den reelle banefeilen i de kringkastede efemeridene.


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
from gnssmultipath.constants import earth_rotation_rate
import pandas as pd

# Alle fire systemene returnerer nå koordinater i ECEF-rammen på mottakstidspunktet,
# så SP3-koordinatene roteres likt for alle.
receiver_position = np.array([x_rec, y_rec, z_rec])
diff_frames = []

for system, system_data in sat_coord.items():
    precise_system = df_precise[df_precise['Satellite'].str.startswith(system)]

    for prn, broadcast_xyz in system_data['position'].items():
        if broadcast_xyz is None:
            continue

        satellite = f'{system}{int(prn):02d}'
        precise_satellite = (precise_system[precise_system['Satellite'] == satellite]
                             .sort_values('Epoch'))
        if precise_satellite.empty:
            continue

        broadcast_xyz = np.asarray(broadcast_xyz, dtype=float)
        sp3_xyz = precise_satellite[['X', 'Y', 'Z']].to_numpy()
        if len(sp3_xyz) != len(broadcast_xyz):
            raise ValueError(f'{satellite}: broadcast og SP3 har ulikt antall epoker')

        travel_time = np.linalg.norm(sp3_xyz - receiver_position, axis=1) / SPEED_OF_LIGHT
        theta = earth_rotation_rate(system) * travel_time
        X_sp3, Y_sp3, Z_sp3 = sp3_xyz.T
        reference_xyz = np.column_stack([
            X_sp3 * np.cos(theta) + Y_sp3 * np.sin(theta),
            -X_sp3 * np.sin(theta) + Y_sp3 * np.cos(theta),
            Z_sp3,
        ])

        diff_frames.append(pd.DataFrame({
            'timestamp': pd.to_datetime(precise_satellite['Epoch']).to_numpy(),
            'system': system,
            'SV': satellite,
            'difference_m': np.linalg.norm(broadcast_xyz - reference_xyz, axis=1),
        }))

df_orbit_diff = (pd.concat(diff_frames, ignore_index=True)
                   .sort_values(['timestamp', 'system', 'SV'])
                   .reset_index(drop=True))

display(df_orbit_diff.groupby('system')['difference_m'].agg(['mean', 'max']).round(3))

# Plot hele tidsserien med ett panel per GNSS-system.
systems = sorted(df_orbit_diff['system'].unique())
system_names = {'G': 'GPS', 'R': 'GLONASS', 'E': 'Galileo', 'C': 'BeiDou'}
fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True)
axes = axes.ravel()

for axis, system in zip(axes, systems):
    system_diff = df_orbit_diff[df_orbit_diff['system'] == system]
    for satellite, satellite_diff in system_diff.groupby('SV', sort=True):
        axis.plot(satellite_diff['timestamp'], satellite_diff['difference_m'],
                  linewidth=0.8, label=satellite)
    axis.set_title(f"{system_names.get(system, system)} "
                   f"(middel {system_diff['difference_m'].mean():,.2f} m)")
    axis.set_ylabel('Avvik [m]')
    axis.grid(True, alpha=0.3)
    axis.legend(fontsize=7, ncol=2, loc='upper right')

for axis in axes[len(systems):]:
    axis.set_visible(False)

fig.supxlabel('Tid')
fig.suptitle('Avvik mellom kringkastede og presise satellittkoordinater')
fig.autofmt_xdate()
fig.tight_layout()
display(fig)
plt.close(fig)


<a id="diverse"></a>

# 3. Diverse – polarplot

Azimut- og elevasjonsvinklene fra seksjon 2 kan sendes rett inn i `make_skyplot`. Under vises
satellittbanene til Galileo, først fra de kringkastede efemeridene og deretter fra SP3-fila.

[Tilbake til innholdsfortegnelsen](#innhold)


In [ ]:
from io import BytesIO
from IPython.display import Image
import gnssmultipath.plot.make_polarplot as polarplot

# Skyplot basert på de kringkastede efemeridene
angles = converter.compute_satellite_azimut_and_elevation_angle(drop_below_horizon=True)
system_code = 'E'
system_name = 'Galileo'

fig = polarplot.make_skyplot(angles[system_code]['azimuth'], angles[system_code]['elevation'],
                             system_name, None, use_tex=False, save=False, return_fig=True)

image = BytesIO()
fig.savefig(image, format='png', dpi=300, bbox_inches='tight')
Image(data=image.getvalue(), width=600)


In [ ]:
# Samme plot, men med de presise banene fra SP3-fila. Resultatet har samme struktur som for de
# kringkastede efemeridene, og mottakerposisjonen oppgis her fordi den bare trengs til vinklene.
angles_sp3 = precise.compute_satellite_azimut_and_elevation_angle((x_rec, y_rec, z_rec), drop_below_horizon=True)

fig = polarplot.make_skyplot(angles_sp3[system_code]['azimuth'], angles_sp3[system_code]['elevation'],
                             system_name, None, use_tex=False, save=False, return_fig=True)

image = BytesIO()
fig.savefig(image, format='png', dpi=300, bbox_inches='tight')
Image(data=image.getvalue(), width=600)
